# Advanced tutorial: Build a multivariable dataset

## Research question

**How are daily streamflow, water temperature, dissolved oxygen, and precipitation related at selected STREAM stations?**

## `streamwq` functions used

- `StreamClient.find_sites()`
- `StreamClient.read_water_quality()`
- `StreamClient.read_streamflow()`
- `StreamClient.read_meteorology()`
- `common_observation_period()`
- `resample_timeseries()`
- `join_timeseries()`

In [ ]:
from streamwq.client import StreamClient
from streamwq.joins import join_timeseries
from streamwq.temporal import common_observation_period, resample_timeseries

stream = StreamClient()

## 1. Select stations with the required observations

In [ ]:
sites = stream.find_sites(
    variables=["WTemp_C", "DO_mgL"],
    streamflow=True,
)
site_ids = sites["STREAM_ID"].head(3).tolist()
sites[sites["STREAM_ID"].isin(site_ids)]

## 2. Retrieve the three time-series datasets

In [ ]:
water_quality = stream.read_water_quality(
    site_ids, ["WTemp_C", "DO_mgL"], quality="approved"
)
streamflow = stream.read_streamflow(site_ids, quality="approved")
meteorology = stream.read_meteorology(site_ids, "APCP_surface")
common_start, common_end = common_observation_period(
    water_quality, streamflow, meteorology
)
print(common_start, common_end)

## 3. Aggregate to a common daily resolution

Temperature, oxygen, and flow use daily means. Precipitation uses a daily sum.

In [ ]:
daily_wq = resample_timeseries(
    water_quality, ["WTemp_C", "DO_mgL"], "daily", "mean"
)
daily_flow = resample_timeseries(streamflow, "Q_m3s", "daily", "mean")
daily_rain = resample_timeseries(meteorology, "APCP_surface", "daily", "sum")

## 4. Join exact daily station/date records

Resampling first gives every dataset the same daily timestamps, allowing the package's exact and validated join to be used.

In [ ]:
analysis_data = join_timeseries(daily_wq, daily_flow)
analysis_data = join_timeseries(analysis_data, daily_rain)
print(f"Analysis-ready rows: {len(analysis_data):,}")
analysis_data.head()

In [ ]:
analysis_data[["WTemp_C", "DO_mgL", "Q_m3s", "APCP_surface"]].corr()